In [3]:
import os, requests
from dotenv import load_dotenv

load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***") # 키 전체가 출력되지 않도록 앞 4자만 확인

F379***


In [18]:
import json

def search_word(q: str, num: int = 10, start: int = 1) -> dict: 
    params = { "key": KEY, "q": q, "req_type": "json", "num": num, "start": start, "type1": "word" } 
    r = requests.get( "https://opendict.korean.go.kr/api/search", params=params, timeout=10 ) 
    r.raise_for_status() 
    return r.json()

data:dict = search_word("김치") 
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])


total: int = data["channel"]["total"] 
items: list[dict] = data["channel"]["item"] 
print(f"총 {total}건, 이 페이지 {len(items)}건") 
for item in items[:5]: 
    word: str = item["word"] 
    pos: str = item.get("pos") or "품사 없음" 
    sense: str = item["sense"][0]["definition"] 
    print(f"{word} ({pos}) --> {sense[:40]}")

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          
총 328건, 이 페이지 10건
김치 (품사 없음) --> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린
김-치 (품사 없음) --> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 
김-치 (품사 없음) --> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南
김치 공장 (품사 없음) --> 김치를 만드는 공장.
김치 보릿고개 (품사 없음) --> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 


(a)강의 자료와 비슷하게 함수를 작성하였다. 검색어와 개수, 시작 위치를 받아 우리말샘 API를 호출하고 JSON 응답을 그대로 반환하도록 하였다.

(b) 하라는대로 넣었다. ensure_ascii=False를 넣지 않으면 한글이 유니코드 형태(\ucxxx)로 출력되어 읽기 어려워진다

(c) 응답 데이터에서 전체 검색 결과 수와 현재 페이지 항목 수를 출력하였다. 품사 정보가 없는 경우 오류가 나지 않도록 dict.get과 기본값을 사용하였다.

타입힌트는 한번 적고 그 이후는ai의 자동완성으로 빠르게 붙였다.

In [ ]:
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥"
]

all_items: list[dict] = []

for word in words:
    data = search_word(word)

    total: int = data["channel"]["total"]
    print(f"{word}: {total}건")

    items: list[dict] = data["channel"].get("item", [])
    all_items.extend(items)

    time.sleep(0.3)



김치: 328건
라면: 86건
만두: 89건
김밥: 39건
국수: 227건
떡볶이: 24건
불고기: 38건
비빔밥: 38건


여러 검색어를 한 번에 처리하기 위해 리스트와 반복문을 사용하였다. Q1에서 만든 함수를 그대로 재사용하고, 결과 개수는 응답 JSON 구조를 보고 토탈 값을 출력하게 했다.
문제에서 하라는 대로 타임슬립을 넣었다.

In [ ]:
from collections import Counter

counter: Counter = Counter()

for item in all_items:
    pos: str = item["sense"][0].get("pos") or "(미상)"
    counter[pos] += 1

for pos, count in counter.most_common(3):
    print(f"{pos}: {count}")

명사: 60
(미상): 19
어미: 1


처음에는 품사가 item 바로 아래 있을 것이라 생각했는데 q1-b 이용해서 JSON 구조를 보니 sense[0]["pos"] 안에 들어 있어 그에 맞게 수정하였다. 품사가 없는 항목도 있어서 get()과 or "(미상)"을 사용해 같이 세도록 작성하였다.

관찰 : 명사가 가장 많이 나왔다. 음식 이름을 검색해서 음식 자체를 가리키는 단어가 많이 포함된 것 같고, 품사가 없는 항목도 일부 있어서 (미상)도 꽤 나온 것 같다.
